이 노트북을 실행하는 데 필요한 라이브러리(표준 라이브러리 제외)
- torch
- numpy
- matplotlib
- transformers

- 실습 기본 환경 설정


In [1]:

# 예제 실행 및 시각화를 위한 공통 라이브러리 로딩

# 코랩 환경 등 깃허브 전체를 clone해서 실습하는 경우가 아니라면 
# 공통 라이브러리를 불러오기 위해서 별도의 과정이 필요하므로 code_reference/README.md 파일을 확인하자.
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

# 시각화 결과를 파일에 저장하지 않음
viz.configure(save_grayscale=False)

# matplotlib 시각화에서 한글 폰트 사용 설정
common.set_korean_plot_env()

# 재현성 보장을 위한 시드 고정
#   여기서 재현성은 '동일 컴퓨터, 동일 버전의 파이썬, 동일 버전의 파이토치' 환경에서 재현이 가능하다는 의미로, 
#   독자의 결과는 저자의 결과와 달라질 수 있다는 점을 밝혀둔다.
#   또한 같은 환경에서도 GPU를 사용하는 경우, 일부 연산의 비결정적 성질로 인해 실행시마다 결과가 조금씩 달라질 수 있다.
SEED = 42
common.set_seed(SEED)

# 실습 환경에 맞는 하드웨어 가속기 장치 객체
device = common.get_device()

CUDA를 사용합니다.


# 12-1 transformers 라이브러리로 LLM 다루기

본 노트북은 본문 12-1절의 코드 예제와 관련 내용을 다룬다. 주요 내용은 다음과 같다.
- `pipeline()`으로 LLM에 질문을 던지는 가장 간단한 방법
- `from_pretrained()`로 모델과 토크나이저를 직접 불러오기
- 한국어 토크나이저의 토큰화 결과와 토크나이저 호출 방식
- 역할과 내용으로 이루어진 대화 메시지를 모델 입력 텐서로 변환하고 답변 생성

## pipeline()으로 질문 던지기

- `pipeline()`에 작업 이름과 모델만 넘기면 토크나이저 준비부터 디코딩까지 한 번에 처리한다.
    - 가장 빠르게 결과를 확인할 수 있지만, 생성 과정을 세밀하게 제어하기는 어렵다.

In [2]:
######################################################################################
# 코드 12-1 - pipeline() 함수로 LLM에 질문 던지기
######################################################################################

from transformers import pipeline

# 텍스트 생성(text-generation) 작업용 한국어 LLM 파이프라인 생성
generator = pipeline('text-generation', model='Bllossom/llama-3.2-Korean-Bllossom-3B')
# 파이프라인을 실행하면 리스트 형태로 결과가 반환됨
result = generator('오픈 소스 모델의 주요 라이선스는 어떤 것이 있어?')
print(result[0]['generated_text'])

/home/crapas/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   6%|▋         | 16/254 [00:00<00:01, 142.90it/s]

Loading weights:  12%|█▏        | 31/254 [00:00<00:03, 70.60it/s] 

Loading weights:  16%|█▌        | 40/254 [00:00<00:03, 67.75it/s]

...

Loading weights: 100%|██████████| 254/254 [00:01<00:00, 139.75it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


오픈 소스 모델의 주요 라이선스는 어떤 것이 있어? 

1. **GNU General Public License (GPL)**: GPL은 오픈 소스 소프트웨어의 기본적인 라이선스로, 소프트웨어의 소스 코드가 공공 도메인에 속하도록 하며, 다른 개발자들이 소프트웨어를 사용하고 수정할 수 있도록 하기 위해 사용됩니다. GPL은 특정 조건을 충족해야 하며, 특정 조건이 충족되지 않으면 GPL의 규칙을 준수해야 합니다. 

2. **MIT License**: MIT License는 오픈 소스 소프트웨어의 라이선스로, 소프트웨어 사용을 허용하고 소스 코드에 대한 접근을 허용하는 간단한 라이선스입니다. MIT License는 사용자에게 소프트웨어 사용 및 배포를 허용하며, 소스 코드를 공개할 필요가 없습니다. 

3. **Apache License**: Apache License는 오픈 소스 소프트웨어의 라이선스로, 소프트웨어 사용을 허용하고 소스 코드에 대한 접근을 허용합니다. Apache License는 특정 조건을 충족해야 하며, 특정 조건이 충족되지 않으면 Apache License의 규칙을


- `pipeline()`이 불러온 모델은 GPU 메모리를 차지한다. 뒤에서 모델을 다시 불러오므로 먼저 정리한다.

In [3]:
# 참고 - 파이프라인이 점유한 메모리 정리
import gc
import torch

del generator, result
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## from_pretrained()로 모델과 토크나이저 불러오기

- 생성 과정을 직접 제어하려면 모델과 토크나이저를 따로 불러온다.
    - 본문 [표 12-1]은 작업 성격에 따라 골라 쓰는 `Auto` 모델 클래스를 정리한다.

In [4]:
######################################################################################
# 코드 12-2 - from_pretrained()로 모델과 토크나이저 불러오기
######################################################################################

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 허깅페이스 허브의 모델 식별자를 그대로 지정
MODEL_NAME = 'Bllossom/llama-3.2-Korean-Bllossom-3B'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# torch_dtype=torch.float16 으로 FP16 로딩 (생략 시 FP32라 메모리 두 배)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16)
model.to(device)

model.eval()
print(f'모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}')
print(f'토크나이저 어휘 크기: {tokenizer.vocab_size:,}')

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/254 [00:00<04:09,  1.01it/s]

Loading weights:  19%|█▉        | 49/254 [00:01<00:03, 61.32it/s]

Loading weights:  30%|███       | 77/254 [00:01<00:03, 48.73it/s]

Loading weights:  37%|███▋      | 94/254 [00:01<00:02, 56.04it/s]

...

Loading weights:  78%|███████▊  | 197/254 [00:03<00:00, 87.63it/s]

Loading weights:  83%|████████▎ | 211/254 [00:03<00:00, 94.60it/s]

Loading weights:  91%|█████████ | 230/254 [00:03<00:00, 114.91it/s]

Loading weights: 100%|██████████| 254/254 [00:03<00:00, 73.60it/s] 

모델 파라미터 수: 3,212,749,824
토크나이저 어휘 크기: 128,000


## 토크나이저

- 토크나이저의 `tokenize()` 메서드는 고유 번호가 아닌 토큰 문자열의 리스트를 반환한다.
    - 한국어는 형태소보다 잘게 쪼개진 서브워드 단위로 나뉘는 경우가 많다.

In [5]:
######################################################################################
# 코드 12-3 - 한국어 토크나이저의 토큰화 결과
######################################################################################

# gogamza/kobart-base-v2 모델: 한국어 BART 모델 (12-2절에서 자세히 소개)
kobart_tokenizer = AutoTokenizer.from_pretrained('gogamza/kobart-base-v2')
text = '검은 소가 누렁소보다 일을 더 잘합니다.'
print(kobart_tokenizer.tokenize(text))

['▁검은', '▁소', '가', '▁누', '렁', '소', '보다', '▁일을', '▁더', '▁잘', '합니다.']


- `tokenize()` 대신 객체 호출(`tokenizer(text)`) 형태로 사용하면 모델에 바로 넣을 수 있는 딕셔너리를 얻는다.
    - `input_ids`(토큰 고유 번호)와 `attention_mask`(패딩이 아닌 위치)를 담고 있다.

In [6]:
######################################################################################
# 코드 12-4 - 토크나이저 호출 방식과 반환값
######################################################################################

text = '검은 소가 누렁소보다 일을 더 잘합니다.'
list_result = kobart_tokenizer(text)                      # 리스트로 결과 반환
pt_result = kobart_tokenizer(text, return_tensors='pt')   # 파이토치 텐서로 반환
print('리스트 반환:')
print(list_result)
print()
print('파이토치 텐서 반환:')
print(pt_result)

리스트 반환:
{'input_ids': [19628, 14081, 8981, 14402, 10295, 11319, 14310, 15462, 14166, 14334, 20357], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

파이토치 텐서 반환:
{'input_ids': tensor([[19628, 14081,  8981, 14402, 10295, 11319, 14310, 15462, 14166, 14334,
         20357]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


## 대화 메시지 구성과 답변 생성

- 지시어 튜닝 모델은 역할(`role`)과 내용(`content`)으로 이루어진 딕셔너리의 리스트를 입력으로 받는다.
    - 역할에는 `system`(모델의 성격과 지침), `user`(사용자 발화), `assistant`(모델의 이전 답변)를 사용한다.

In [7]:
######################################################################################
# 코드 12-5 - 대화틀 함수
######################################################################################

def chat_template(prompt):
    message = [
        {'role': 'system',
         'content': '당신은 한국어를 사용하는 친절한 AI 친구입니다.'},
        {'role': 'user', 'content': prompt},
    ]
    return message


example_messages = chat_template('안녕? 오늘 날씨가 좋구나!')
print(example_messages)

[{'role': 'system', 'content': '당신은 한국어를 사용하는 친절한 AI 친구입니다.'}, {'role': 'user', 'content': '안녕? 오늘 날씨가 좋구나!'}]


- 대화틀로 만든 메시지는 토크나이저의 `apply_chat_template()`을 거쳐 모델 입력 텐서가 된다.
    - 모델마다 대화를 표현하는 특수 토큰 형식이 다르므로, 이 메서드가 모델에 맞는 형식으로 변환해 준다.

In [8]:
######################################################################################
# 코드 12-6 - 대화 메시지를 모델 입력 텐서로 변환
######################################################################################

def generate_message(prompt, device):
    messages = chat_template(prompt)
    # apply_chat_template은 모델 학습 형식대로 입력을 재구성한다.
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=False,                 # input_ids 텐서만 반환
    ).to(device)
    return input_ids


example_input_ids = generate_message('안녕? 오늘 날씨가 좋구나!', device)
print(f'입력 텐서 shape: {example_input_ids.shape}')   # (1, S)
print(f'디코딩 결과:\n{tokenizer.decode(example_input_ids[0])}')

입력 텐서 shape: torch.Size([1, 61])
디코딩 결과:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Sep 2026

당신은 한국어를 사용하는 친절한 AI 친구입니다.<|eot_id|><|start_header_id|>user<|end_header_id|>

안녕? 오늘 날씨가 좋구나!<|eot_id|><|start_header_id|>assistant<|end_header_id|>




- 모델의 `generate()` 메서드에 입력 텐서를 넣으면 답변 토큰이 생성된다.
    - 본문 [표 12-2]는 `generate()` 메서드의 주요 인자를 정리한다.

In [9]:
######################################################################################
# 코드 12-7 - LLM에 대화 메시지를 입력해 답변 생성
######################################################################################

# 모델이 사용하는 종료 특수 토큰 두 종류를 모두 종료 토큰으로 지정
terminators = [
    tokenizer.convert_tokens_to_ids('<|end_of_text|>'),
    tokenizer.convert_tokens_to_ids('<|eot_id|>'),
]
prompt = '안녕? 오늘 날씨가 좋구나!'
max_new_tokens = 512
input_ids = generate_message(prompt, device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,  # 최대 생성 토큰 길이
        eos_token_id=terminators,
        do_sample=True,                 # 확률에 기반해 토큰 샘플링
        temperature=0.6,                # 생성 온도 (낮을수록 보수적)
        top_p=0.9,                      # 누적 확률 90% 안의 후보에서 선택
        pad_token_id=tokenizer.eos_token_id,
    )

# 출력 텐서에는 입력 토큰까지 포함되므로 입력 길이만큼 잘라 새 토큰만 디코딩
llm_generated = output_ids[0][input_ids.shape[-1]:]
result_text = tokenizer.decode(llm_generated, skip_special_tokens=True)
print(f'사용자 프롬프트 : {prompt}')
print(f'LLM의 답변 : {result_text}')

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


사용자 프롬프트 : 안녕? 오늘 날씨가 좋구나!
LLM의 답변 : 안녕하세요! 날씨가 좋네요, 좋은 하루 되세요!


## 정리

- `pipeline()`은 가장 간단한 사용법이고, `from_pretrained()`로 모델과 토크나이저를 직접 불러오면 생성 과정을 제어할 수 있다.
- 지시어 튜닝 모델에는 역할과 내용으로 구성한 대화 메시지를 `apply_chat_template()`으로 변환해 넣는다.
- `generate()` 메서드의 인자로 생성 길이, 샘플링 여부, 반복 억제 등을 조절한다.